In [1]:

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import LSTM


from tensorflow.keras.layers import (
    Conv1D,
    Dense,
    Dropout,
    GlobalMaxPooling1D
)

from tensorflow.keras.optimizers import Adam
from scipy.stats import t


C:\Users\USER\anaconda3\envs\py310\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


In [2]:
import sys
import tensorflow as tf

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

Python: C:\Users\USER\anaconda3\envs\py310\python.exe
TensorFlow: 2.10.1
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

DATASET_PATH = "PhiUSIIL_Phishing_URL_Dataset.csv"

BATCH_SIZE = 16
LEARNING_RATE = 0.001
EPOCHS = 30
SEEDS = [42, 3, 7, 72, 82]

# ------------------------------------------------------------
# REPRODUCIBILITY
# ------------------------------------------------------------

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)

# ------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------

data = pd.read_csv(DATASET_PATH)

print("Dataset Shape:", data.shape)
print(data.head())

print("\nColumns:")
print(data.columns.tolist())

# ------------------------------------------------------------
# VERIFY TARGET COLUMN
# ------------------------------------------------------------

TARGET = "label"

if TARGET not in data.columns:
    raise ValueError(f"{TARGET} not found in dataset.")

# ------------------------------------------------------------
# HANDLE MISSING VALUES
# ------------------------------------------------------------

numeric_cols = data.select_dtypes(include=np.number).columns.tolist()
categorical_cols = data.select_dtypes(exclude=np.number).columns.tolist()

# Remove target from feature lists
if TARGET in numeric_cols:
    numeric_cols.remove(TARGET)

if TARGET in categorical_cols:
    categorical_cols.remove(TARGET)

# Fill missing values
data[numeric_cols] = data[numeric_cols].fillna(
    data[numeric_cols].mean()
)

data[categorical_cols] = data[categorical_cols].fillna("unknown")

# ------------------------------------------------------------
# ENCODE CATEGORICAL FEATURES
# ------------------------------------------------------------

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col].astype(str))

# ------------------------------------------------------------
# ENCODE TARGET IF NEEDED
# ------------------------------------------------------------

if data[TARGET].dtype == object:
    target_encoder = LabelEncoder()
    data[TARGET] = target_encoder.fit_transform(data[TARGET])

# ------------------------------------------------------------
# CHECK CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\nClass Distribution:")
print(data[TARGET].value_counts())

# Remove classes having fewer than 2 samples
class_counts = data[TARGET].value_counts()

valid_classes = class_counts[class_counts >= 2].index

data = data[data[TARGET].isin(valid_classes)].reset_index(drop=True)

print("\nAfter Removing Rare Classes:")
print(data[TARGET].value_counts())

# ------------------------------------------------------------
# FEATURES & LABELS
# ------------------------------------------------------------

X = data.drop(columns=[TARGET])
y = data[TARGET]

# ------------------------------------------------------------
# TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ------------------------------------------------------------
# NORMALIZATION
# ------------------------------------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ------------------------------------------------------------
# RESHAPE FOR CNN
# ------------------------------------------------------------

X_train = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("\nTraining Samples :", len(X_train))
print("Testing Samples  :", len(X_test))

print("\nTraining Shape :", X_train.shape)
print("Testing Shape  :", X_test.shape)

print("\nDone.")

Dataset Shape: (235795, 56)
     FILENAME                                 URL  URLLength  \
0  521848.txt    https://www.southbankmosaics.com         31   
1   31372.txt            https://www.uni-mainz.de         23   
2  597387.txt      https://www.voicefmradio.co.uk         29   
3  554095.txt         https://www.sfnmjournal.com         26   
4  151578.txt  https://www.rewildingargentina.org         33   

                       Domain  DomainLength  IsDomainIP  TLD  \
0    www.southbankmosaics.com            24           0  com   
1            www.uni-mainz.de            16           0   de   
2      www.voicefmradio.co.uk            22           0   uk   
3         www.sfnmjournal.com            19           0  com   
4  www.rewildingargentina.org            26           0  org   

   URLSimilarityIndex  CharContinuationRate  TLDLegitimateProb  ...  Pay  \
0               100.0              1.000000           0.522907  ...    0   
1               100.0              0.666667       

# CNN

In [4]:



def create_model(input_shape):

    model = Sequential()

    model.add(
        Input(shape=input_shape)
    )

    model.add(
        Conv1D(
            filters=64,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Conv1D(
            filters=32,
            kernel_size=3,
            activation="relu"
        )
    )

    model.add(
        GlobalMaxPooling1D()
    )

    model.add(
        Dense(
            32,
            activation="relu"
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )

    return model

In [5]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (1D-CNN)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # Build 1D-CNN Model
    model = create_model(
        (X_train.shape[1], X_train.shape[2])
    )

    # Train Model
    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )

    # =====================================================
    # Test Loss
    # =====================================================

    evaluation = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    test_loss = evaluation[0]

    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()

    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42


I0000 00:00:1786005594.324849      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/30
   57/11790 ━━━━━━━━━━━━━━━━━━━━ 32s 3ms/step - accuracy: 0.5994 - auc: 0.6243 - loss: 0.6736  

I0000 00:00:1786005601.182959     135 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


11790/11790 ━━━━━━━━━━━━━━━━━━━━ 49s 4ms/step - accuracy: 0.9651 - auc: 0.9933 - loss: 0.0992 - val_accuracy: 0.9919 - val_auc: 0.9997 - val_loss: 0.0307
Epoch 2/30
11790/11790 ━━━━━━━━━━━━━━━━━━━━ 40s 3ms/step - accuracy: 0.9919 - auc: 0.9991 - loss: 0.0257 - val_accuracy: 0.9980 - val_auc: 0.9999 - val_loss: 0.0061
Epoch 3/30
11790/11790 ━━━━━━━━━━━━━━━━━━━━ 39s 3ms/step - accuracy: 0.9948 - auc: 0.9994 - loss: 0.0170 - val_accuracy: 0.9983 - val_auc: 0.9999 - val_loss: 0.0059
Epoch 4/30
11790/11790 ━━━━━━━━━━━━━━━━━━━━ 39s 3ms/step - accuracy: 0.9959 - auc: 0.9995 - loss: 0.0137 - val_accuracy: 0.9989 - val_auc: 0.9999 - val_loss: 0.0037
Epoch 5/30
11790/11790 ━━━━━━━━━━━━━━━━━━━━ 38s 3ms/step - accuracy: 0.9961 - auc: 0.9995 - loss: 0.0129 - val_accuracy: 0.9992 - val_auc: 0.9999 - val_loss: 0.0029
Epoch 6/30
11790/11790 ━━━━━━━━━━━━━━━━━━━━ 38s 3ms/step - accuracy: 0.9968 - auc: 0.9996 - loss: 0.0117 - val_accuracy: 0.9992 - val_auc: 1.0000 - val_loss: 0.0023
Epoch 7/30
11790/1179

In [6]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.001682  0.999597   0.999481  0.999815  0.999648  0.999177   
1     3  0.001074  0.999682   0.999926  0.999518  0.999722  0.999351   
2     7  0.000999  0.999703   0.999815  0.999666  0.999740  0.999394   
3    72  0.000827  0.999682   0.999518  0.999926  0.999722  0.999350   
4    82  0.001471  0.999576   0.999370  0.999889  0.999629  0.999134   

        AUC  Specificity  
0  0.999963     0.999307  
1  0.999999     0.999901  
2  0.999999     0.999752  
3  1.000000     0.999356  
4  0.999973     0.999158  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD            95% CI
0         Loss  0.001211  0.000354  0.0012 ± 0.0004  [0.0008, 0.0016]
1     Accuracy  0.999648  0.000057  0.9996 ± 0.0001  [0.9996, 0.9997]
2    Precision  0.999622  0.000236  0.9996 ± 0.0002  [0.9993, 0.9999]
3       Recall  0.999763  0.000169  0.9998 ± 0.0002  [0.9996, 1.0000]
4

# LSTM

In [4]:



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        LSTM(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        LSTM(
            64
        )
    )

    model.add(
        Dropout(0.5
        )
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [5]:
# ============================================================
# PART 2
# CENTRALIZED TRAINING (LSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build LSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
11790/11790 [==============================] - 302s 25ms/step - loss: 0.0247 - accuracy: 0.9918 - auc: 0.9993 - val_loss: 0.0204 - val_accuracy: 0.9955 - val_auc: 0.9979
Epoch 2/30
11790/11790 [==============================] - 300s 25ms/step - loss: 0.0081 - accuracy: 0.9976 - auc: 0.9997 - val_loss: 0.0035 - val_accuracy: 0.9988 - val_auc: 1.0000
Epoch 3/30
11790/11790 [==============================] - 301s 26ms/step - loss: 0.0041 - accuracy: 0.9989 - auc: 0.9998 - val_loss: 0.0064 - val_accuracy: 0.9980 - val_auc: 0.9999
Epoch 4/30
11790/11790 [==============================] - 300s 25ms/step - loss: 0.0029 - accuracy: 0.9992 - auc: 0.9999 - val_loss: 0.0020 - val_accuracy: 0.9994 - val_auc: 1.0000
Epoch 5/30
11790/11790 [==============================] - 302s 26ms/step - loss: 0.0023 - accuracy: 0.9994 - auc: 0.9999 - val_loss: 0.0010 - val_accuracy: 0.9998 - val_auc: 1.0000
Epoch 6/30
11790/11790 [==============================] - 305s 26ms/step - lo

In [6]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  AUC  \
0    42  0.001159  0.999703   1.000000  0.999481  0.999740  0.999394  1.0   
1     3  0.000020  1.000000   1.000000  1.000000  1.000000  1.000000  1.0   
2     7  0.000095  0.999958   0.999926  1.000000  0.999963  0.999913  1.0   
3    72  0.000082  0.999958   0.999926  1.000000  0.999963  0.999913  1.0   
4    82  0.000589  0.999894   0.999815  1.000000  0.999907  0.999783  1.0   

   Specificity  
0     1.000000  
1     1.000000  
2     0.999901  
3     0.999901  
4     0.999752  


Performance Summary (5 Seeds)
        Metric      Mean            SD        Mean ± SD             95% CI
0         Loss  0.000389  4.871781e-04  0.0004 ± 0.0005  [-0.0002, 0.0010]
1     Accuracy  0.999902  1.176822e-04  0.9999 ± 0.0001   [0.9998, 1.0000]
2    Precision  0.999933  7.597409e-05  0.9999 ± 0.0001   [0.9998, 1.0000]
3       Recall  0.999896  2.321465e-04  0.9999 ± 0.0002   [0.9996, 1.0002]
4     

# BiLSTM

In [5]:

from tensorflow.keras.layers import Bidirectional



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            LSTM(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            LSTM(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [6]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiLSTM)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
11790/11790 [==============================] - 156s 13ms/step - loss: 0.0125 - accuracy: 0.9960 - auc: 0.9998 - val_loss: 0.0053 - val_accuracy: 0.9984 - val_auc: 0.9999
Epoch 2/30
11790/11790 [==============================] - 151s 13ms/step - loss: 0.0033 - accuracy: 0.9990 - auc: 0.9999 - val_loss: 6.1088e-04 - val_accuracy: 0.9998 - val_auc: 1.0000
Epoch 3/30
11790/11790 [==============================] - 148s 13ms/step - loss: 0.0019 - accuracy: 0.9993 - auc: 0.9999 - val_loss: 6.3497e-04 - val_accuracy: 0.9997 - val_auc: 1.0000
Epoch 4/30
11790/11790 [==============================] - 148s 13ms/step - loss: 0.0020 - accuracy: 0.9995 - auc: 0.9999 - val_loss: 4.8487e-04 - val_accuracy: 0.9999 - val_auc: 1.0000
Epoch 5/30
11790/11790 [==============================] - 149s 13ms/step - loss: 0.0012 - accuracy: 0.9997 - auc: 1.0000 - val_loss: 3.1401e-04 - val_accuracy: 0.9999 - val_auc: 1.0000
Epoch 6/30
11790/11790 [==============================] - 149

In [7]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",      
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  AUC  \
0    42  0.000043  0.999979   0.999963  1.000000  0.999981  0.999957  1.0   
1     3  0.000284  0.999936   0.999926  0.999963  0.999944  0.999870  1.0   
2     7  0.000433  0.999873   0.999815  0.999963  0.999889  0.999740  1.0   
3    72  0.000002  1.000000   1.000000  1.000000  1.000000  1.000000  1.0   
4    82  0.000363  0.999936   0.999926  0.999963  0.999944  0.999870  1.0   

   Specificity  
0     0.999950  
1     0.999901  
2     0.999752  
3     1.000000  
4     0.999901  


Performance Summary (5 Seeds)
        Metric      Mean            SD        Mean ± SD             95% CI
0         Loss  0.000225  1.928835e-04  0.0002 ± 0.0002  [-0.0000, 0.0005]
1     Accuracy  0.999945  4.881725e-05  0.9999 ± 0.0000   [0.9999, 1.0000]
2    Precision  0.999926  6.935619e-05  0.9999 ± 0.0001   [0.9998, 1.0000]
3       Recall  0.999978  2.030859e-05  1.0000 ± 0.0000   [1.0000, 1.0000]
4     

# GRU

In [8]:

from tensorflow.keras.layers import GRU



def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        GRU(
            128,
            return_sequences=True
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        GRU(
            64
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [9]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (GRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
11790/11790 [==============================] - 82s 7ms/step - loss: 0.0246 - accuracy: 0.9916 - auc: 0.9993 - val_loss: 0.0043 - val_accuracy: 0.9987 - val_auc: 0.9997
Epoch 2/30
11790/11790 [==============================] - 83s 7ms/step - loss: 0.0047 - accuracy: 0.9987 - auc: 0.9998 - val_loss: 0.0017 - val_accuracy: 0.9995 - val_auc: 0.9999
Epoch 3/30
11790/11790 [==============================] - 82s 7ms/step - loss: 0.0030 - accuracy: 0.9992 - auc: 0.9999 - val_loss: 0.0016 - val_accuracy: 0.9994 - val_auc: 1.0000
Epoch 4/30
11790/11790 [==============================] - 82s 7ms/step - loss: 0.0024 - accuracy: 0.9994 - auc: 0.9999 - val_loss: 8.1199e-04 - val_accuracy: 0.9997 - val_auc: 1.0000
Epoch 5/30
11790/11790 [==============================] - 80s 7ms/step - loss: 0.0019 - accuracy: 0.9994 - auc: 0.9999 - val_loss: 7.7217e-04 - val_accuracy: 0.9998 - val_auc: 1.0000
Epoch 6/30
11790/11790 [==============================] - 82s 7ms/step - loss: 

In [10]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  AUC  \
0    42  0.000232  0.999936   1.000000  0.999889  0.999944  0.999870  1.0   
1     3  0.000579  0.999915   0.999852  1.000000  0.999926  0.999827  1.0   
2     7  0.000153  0.999915   0.999926  0.999926  0.999926  0.999827  1.0   
3    72  0.000317  0.999873   1.000000  0.999778  0.999889  0.999740  1.0   
4    82  0.000593  0.999873   0.999778  1.000000  0.999889  0.999740  1.0   

   Specificity  
0     1.000000  
1     0.999802  
2     0.999901  
3     1.000000  
4     0.999703  


Performance Summary (5 Seeds)
        Metric      Mean            SD        Mean ± SD            95% CI
0         Loss  0.000375  2.014543e-04  0.0004 ± 0.0002  [0.0001, 0.0006]
1     Accuracy  0.999902  2.844931e-05  0.9999 ± 0.0000  [0.9999, 0.9999]
2    Precision  0.999911  9.666776e-05  0.9999 ± 0.0001  [0.9998, 1.0000]
3       Recall  0.999918  9.232406e-05  0.9999 ± 0.0001  [0.9998, 1.0000]
4          

# BiGRU

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, GRU, Bidirectional, Dropout, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf


def create_model():

    model = Sequential()

    model.add(
        Input(
            shape=(
                X_train.shape[1],
                X_train.shape[2]
            )
        )
    )

    model.add(
        Bidirectional(
            GRU(
                128,
                return_sequences=True
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Bidirectional(
            GRU(
                64
            )
        )
    )

    model.add(
        Dropout(0.5)
    )

    model.add(
        Dense(
            1,
            activation='sigmoid'
        )
    )

    model.compile(
        optimizer=Adam(
            learning_rate=LEARNING_RATE
        ),

        loss='binary_crossentropy',

        metrics=[
            'accuracy',
            tf.keras.metrics.AUC(name='auc')
        ]
    )

    return model

In [12]:
# ============================================================
# PART 2
# CENTRALIZED BiTRAINING (BiGRU)
# ============================================================

results = []

for seed in SEEDS:

    print("=" * 70)
    print(f"Running Seed : {seed}")
    print("=" * 70)

    # Reproducibility
    set_seed(seed)

    # =====================================================
    # Build BiLSTM Model
    # =====================================================

    model = create_model()


    # =====================================================
    # Train Model
    # =====================================================

    history = model.fit(
        X_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_test, y_test),
        verbose=1
    )


    # =====================================================
    # Test Loss
    # =====================================================

    test_loss, test_accuracy, test_auc = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )


    # =====================================================
    # Prediction
    # =====================================================

    y_prob = model.predict(
        X_test,
        verbose=0
    ).flatten()

    y_pred = (y_prob >= 0.5).astype(int)


    # =====================================================
    # Performance Metrics
    # =====================================================

    accuracy = accuracy_score(
        y_test,
        y_pred
    )

    precision = precision_score(
        y_test,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        zero_division=0
    )

    mcc = matthews_corrcoef(
        y_test,
        y_pred
    )

    auc = roc_auc_score(
        y_test,
        y_prob
    )


    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred
    ).ravel()


    specificity = tn / (tn + fp)


    # =====================================================
    # Save Results
    # =====================================================

    results.append({

        "Seed": seed,

        "Loss": test_loss,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1": f1,

        "MCC": mcc,

        "AUC": auc,

        "Specificity": specificity

    })


    # =====================================================
    # Print Results
    # =====================================================

    print("\nTest Performance")
    print("-" * 40)

    print(f"Loss        : {test_loss:.4f}")
    print(f"Accuracy    : {accuracy:.4f}")
    print(f"Precision   : {precision:.4f}")
    print(f"Recall      : {recall:.4f}")
    print(f"F1-score    : {f1:.4f}")
    print(f"MCC         : {mcc:.4f}")
    print(f"AUC         : {auc:.4f}")
    print(f"Specificity : {specificity:.4f}")

    print()

Running Seed : 42
Epoch 1/30
11790/11790 [==============================] - 145s 12ms/step - loss: 0.0077 - accuracy: 0.9975 - auc: 0.9999 - val_loss: 0.0048 - val_accuracy: 0.9983 - val_auc: 0.9999
Epoch 2/30
11790/11790 [==============================] - 140s 12ms/step - loss: 0.0025 - accuracy: 0.9991 - auc: 0.9999 - val_loss: 0.0022 - val_accuracy: 0.9993 - val_auc: 0.9999
Epoch 3/30
11790/11790 [==============================] - 143s 12ms/step - loss: 0.0016 - accuracy: 0.9995 - auc: 0.9999 - val_loss: 7.1239e-04 - val_accuracy: 0.9998 - val_auc: 1.0000
Epoch 4/30
11790/11790 [==============================] - 141s 12ms/step - loss: 0.0014 - accuracy: 0.9996 - auc: 1.0000 - val_loss: 4.3494e-04 - val_accuracy: 0.9998 - val_auc: 1.0000
Epoch 5/30
11790/11790 [==============================] - 140s 12ms/step - loss: 0.0014 - accuracy: 0.9996 - auc: 1.0000 - val_loss: 3.3435e-04 - val_accuracy: 0.9999 - val_auc: 1.0000
Epoch 6/30
11790/11790 [==============================] - 156s 13

In [13]:
# ============================================================
# MEAN ± SD + 95% CONFIDENCE INTERVAL
# ============================================================

results_df = pd.DataFrame(results)

print("\nResults from 5 Seeds")
print(results_df)


# ------------------------------------------------------------
# Function to calculate Mean, SD and 95% CI
# ------------------------------------------------------------

def calculate_statistics(values):

    values = np.array(values)

    n = len(values)

    mean = np.mean(values)

    sd = np.std(values, ddof=1)

    se = sd / np.sqrt(n)

    t_value = t.ppf(0.975, df=n-1)

    margin = t_value * se

    lower = mean - margin

    upper = mean + margin

    return mean, sd, lower, upper


# ------------------------------------------------------------
# Compute statistics for each metric
# ------------------------------------------------------------

summary = []

metrics = [
    "Loss",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "MCC",
    "AUC",
    "Specificity"
]


for metric in metrics:

    mean, sd, lower, upper = calculate_statistics(
        results_df[metric]
    )

    summary.append({

        "Metric": metric,

        "Mean": mean,

        "SD": sd,

        "Mean ± SD": f"{mean:.4f} ± {sd:.4f}",

        "95% CI": f"[{lower:.4f}, {upper:.4f}]"

    })


summary_df = pd.DataFrame(summary)


# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\n")
print("="*70)
print("Performance Summary (5 Seeds)")
print("="*70)

print(summary_df)


# ------------------------------------------------------------
# Save Results
# ------------------------------------------------------------

results_df.to_csv(
    "Centralized_5Seed_Results.csv",
    index=False
)

summary_df.to_csv(
    "Centralized_Performance_Summary.csv",
    index=False
)


print("\nCSV files saved successfully.")


Results from 5 Seeds
   Seed      Loss  Accuracy  Precision    Recall        F1       MCC  \
0    42  0.000501  0.999830   0.999778  0.999926  0.999852  0.999654   
1     3  0.001534  0.999576   0.999518  0.999740  0.999629  0.999134   
2     7  0.009355  0.997731   0.999182  0.996848  0.998014  0.995372   
3    72  0.001549  0.999406   0.999777  0.999184  0.999481  0.998788   
4    82  0.002110  0.999491   0.999148  0.999963  0.999555  0.998961   

        AUC  Specificity  
0  1.000000     0.999703  
1  0.999998     0.999356  
2  0.999971     0.998910  
3  0.999998     0.999703  
4  0.999999     0.998861  


Performance Summary (5 Seeds)
        Metric      Mean        SD        Mean ± SD             95% CI
0         Loss  0.003010  0.003594  0.0030 ± 0.0036  [-0.0015, 0.0075]
1     Accuracy  0.999207  0.000840  0.9992 ± 0.0008   [0.9982, 1.0003]
2    Precision  0.999481  0.000307  0.9995 ± 0.0003   [0.9991, 0.9999]
3       Recall  0.999132  0.001314  0.9991 ± 0.0013   [0.9975, 1.00